# RS-VLM — Phase 1: MAE Pretraining

Trains the **CNN-ViT hybrid encoder** on EuroSAT using Masked Autoencoding.

- **What trains:** CNNStem + ViTBody + GSDAdapter + MAEDecoder (decoder is discarded after)
- **What's frozen:** nothing — full encoder trains
- **Dataset:** EuroSAT RGB (27,000 satellite images, 10 classes)
- **Objective:** reconstruct 75% masked CNN feature map patches
- **Output:** `encoder_final.pt` saved to Google Drive

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Mount Google Drive
Checkpoints will be saved here so they survive Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/rs_vlm/checkpoints/phase1', exist_ok=True)
print('Drive mounted. Checkpoint dir ready.')

## 2. Clone repo & install dependencies

In [ ]:
# --- clone your repo ---
# replace with your actual GitHub URL
!git clone https://github.com/YOUR_USERNAME/rs_vlm.git /content/rs_vlm
%cd /content/rs_vlm

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## 3. Download EuroSAT dataset

~90MB zip. Downloads directly from the official DFKI source.

In [ ]:
import os

EUROSAT_DIR = '/content/rs_vlm/data/EuroSAT'

if not os.path.exists(EUROSAT_DIR):
    print('Downloading EuroSAT...')
    !wget -q --show-progress https://madm.dfki.de/files/sentinel/EuroSAT.zip -O /tmp/EuroSAT.zip
    !unzip -q /tmp/EuroSAT.zip -d /content/rs_vlm/data/
    # the zip extracts to a '2750' folder — rename it to EuroSAT
    if os.path.exists('/content/rs_vlm/data/2750'):
        os.rename('/content/rs_vlm/data/2750', EUROSAT_DIR)
    print(f'EuroSAT ready at {EUROSAT_DIR}')
else:
    print('EuroSAT already downloaded.')

# quick sanity check
classes = os.listdir(EUROSAT_DIR)
print(f'Classes ({len(classes)}): {classes}')

## 4. Smoke test — verify encoder + MAE pipeline before full training

In [ ]:
import sys
sys.path.insert(0, '/content/rs_vlm')

import torch
from encoder.hybrid_encoder import HybridEncoder
from training.phase1_pretrain import MAEDecoder, mae_loss
from data.eurosat import mae_mask_patches

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

encoder = HybridEncoder(cnn_pretrained=False).to(device)
decoder = MAEDecoder().to(device)

dummy_imgs = torch.randn(2, 3, 224, 224).to(device)
dummy_gsd  = torch.tensor([10.0, 0.3]).to(device)

feat_map             = encoder.cnn_stem(dummy_imgs)
masked_map, mask     = mae_mask_patches(feat_map, mask_ratio=0.75)
gsd_bias             = encoder.gsd_adapter(dummy_gsd)
tokens               = encoder.vit_body(masked_map, gsd_bias=gsd_bias)
reconstructed        = decoder(tokens)
loss                 = mae_loss(feat_map.detach(), reconstructed, mask)

print(f'feat_map:      {feat_map.shape}')
print(f'masked_map:    {masked_map.shape}')
print(f'tokens:        {tokens.shape}')
print(f'reconstructed: {reconstructed.shape}')
print(f'MAE loss:      {loss.item():.4f}')
assert reconstructed.shape == feat_map.shape
print('\nSmoke test passed — ready for full training!')

## 5. Run Phase 1 training

In [ ]:
from training.phase1_pretrain import train_phase1

encoder = train_phase1(
    config_path='configs/colab_config.yaml',
    resume_from=None,  # set to a checkpoint path to resume, e.g. '/content/drive/MyDrive/rs_vlm/checkpoints/phase1/encoder_epoch005.pt'
)

## 6. Verify final checkpoint

In [ ]:
import torch

ckpt_path = '/content/drive/MyDrive/rs_vlm/checkpoints/phase1/encoder_final.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')

print(f"Saved at epoch: {ckpt['epoch']}")
print(f"Final MAE loss: {ckpt['loss']:.4f}")
print(f"Keys in checkpoint: {list(ckpt.keys())}")
print('\nPhase 1 complete. Use encoder_final.pt as input to Phase 2.')

## 7. (Optional) Plot training loss

If you want to log losses during training, you can modify `train_phase1` to return a loss list, or add a simple logger. For now this cell is a placeholder.

In [ ]:
# placeholder — add loss logging to train_phase1 if needed
print('Add loss history logging to train_phase1() to plot here.')